# 01 - Training

Training is the same call the CLI makes: `apexgpt.features.training.service.train`.
Nothing here is notebook-only logic, so a notebook run and
`python -m apexgpt train` produce the same checkpoint.

In [ ]:
import sys
from pathlib import Path

# Works whether ApexGPT is pip-installed or merely cloned next to this notebook.
def _repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "apexgpt" / "__init__.py").exists():
            return candidate
    return start

ROOT = _repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import apexgpt
print("apexgpt", apexgpt.__version__, "from", Path(apexgpt.__file__).parent)

In [ ]:
from apexgpt.core.config import Config
from apexgpt.core.paths import RUNS_DIR

cfg = Config()
cfg.data.select_dataset("shakespeare")     # 338k tokens: trains in minutes on a CPU
print("corpus        :", cfg.data.dataset)
print("corpus ready  :", cfg.data.is_ready())
print("token binaries:", cfg.data.binary_dir)
print("run directory :", RUNS_DIR)

## The corpus

Training needs tokenised text. tiny Shakespeare is the fast one - 1.1 MB and
338,025 tokens - so it is the default here. Wikipedia is 1.3 GB and takes about
15 minutes to prepare. Either way the preparation is the same code:

```bash
python -m apexgpt data prepare --source shakespeare
python -m apexgpt data sources             # what else is available
```

In [ ]:
# from apexgpt.features.data.service import prepare
# prepare(cfg.data, source="shakespeare")    # ~4 s: download, tokenize, split 80/20

## Train

`preset="auto"` (the default) sizes the run to this machine. On a CPU-only
laptop that is `cpu-tiny`; the scan in notebook 00 already told you which.

In [ ]:
from apexgpt.features.training.service import train

result = train(cfg, preset="auto", max_iters=100, log_interval=10,
               run_name="gpt-shakespeare")

## Loss history

In [ ]:
for row in (result.history or [])[-10:]:
    print(f"step {row['step']:>5}  train {row['train_loss']:.4f}  "
          f"val {row['val_loss']:.4f}  lr {row['lr']:.2e}")

print("best val:", result.best_val_loss)
print("checkpoint:", result.checkpoint)

## Loss curves

10.82 is `ln(50257)` - the loss of random guessing. Watch it fall.

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(result.checkpoint.parent / "loss_curves.png")))